# 53 — Classification Fine-Tuning Lab (Prompting vs SFT)

## Goal
Compare **zero-shot prompting** vs **SFT fine-tuning** for intent classification.
See when fine-tuning is worth it and when prompting is enough.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Zero-shot prompting** | Classify with instructions only (no training) |
| **SFT for classification** | Train on labeled examples |
| **Head-to-head eval** | Accuracy, latency, cost comparison |

## Stack
- `transformers` + `peft` + `trl`
- Ollama for zero-shot baseline
- Model: **Qwen2.5-0.5B-Instruct** for SFT

In [ ]:
import os, warnings, json, time
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## Step 1 — Intent Classification Dataset

Customer support intents. Each message maps to one of 6 categories.

In [ ]:
INTENTS = ["billing", "technical_issue", "account_access", "feature_request", "cancellation", "general_inquiry"]

labeled_data = [
    # billing
    ("I was charged twice this month", "billing"),
    ("Why is my invoice higher than usual?", "billing"),
    ("Can I get a receipt for last month's payment?", "billing"),
    ("The payment didn't go through but my card was charged", "billing"),
    ("I need to update my credit card information", "billing"),
    ("When will I be charged for the next cycle?", "billing"),
    ("Is there a discount for annual billing?", "billing"),
    ("My refund hasn't shown up yet", "billing"),
    # technical_issue
    ("The app crashes when I upload files", "technical_issue"),
    ("Dashboard is loading very slowly today", "technical_issue"),
    ("Getting a 500 error on the API", "technical_issue"),
    ("The export feature isn't working", "technical_issue"),
    ("My data sync is stuck at 50%", "technical_issue"),
    ("Charts are showing wrong numbers", "technical_issue"),
    ("The mobile view is completely broken", "technical_issue"),
    ("Webhook notifications stopped working", "technical_issue"),
    # account_access
    ("I can't log in to my account", "account_access"),
    ("I forgot my password and the reset email isn't coming", "account_access"),
    ("My account was locked after too many attempts", "account_access"),
    ("How do I enable two-factor authentication?", "account_access"),
    ("I need to change the email on my account", "account_access"),
    ("Can I merge my two accounts into one?", "account_access"),
    # feature_request
    ("It would be great to have dark mode", "feature_request"),
    ("Can you add Slack integration?", "feature_request"),
    ("I wish I could customize the report templates", "feature_request"),
    ("Do you plan to support SSO?", "feature_request"),
    ("Could you add a Gantt chart view?", "feature_request"),
    ("It would help to have keyboard shortcuts", "feature_request"),
    # cancellation
    ("I want to cancel my subscription", "cancellation"),
    ("How do I close my account?", "cancellation"),
    ("I'm switching to a competitor", "cancellation"),
    ("Please stop charging me, I don't use this anymore", "cancellation"),
    ("What happens to my data if I cancel?", "cancellation"),
    # general_inquiry
    ("What's the difference between Pro and Enterprise?", "general_inquiry"),
    ("Do you have any tutorials?", "general_inquiry"),
    ("How many users can I add on my plan?", "general_inquiry"),
    ("Is there a free trial available?", "general_inquiry"),
    ("What are your support hours?", "general_inquiry"),
    ("Do you offer on-premise deployment?", "general_inquiry"),
]

# Split: first N-2 per category for train, last 2 per category for test
from collections import defaultdict
by_intent = defaultdict(list)
for text, label in labeled_data:
    by_intent[label].append((text, label))

train_set, test_set = [], []
for intent, examples in by_intent.items():
    train_set.extend(examples[:-2])
    test_set.extend(examples[-2:])

print(f"Train: {len(train_set)} examples")
print(f"Test:  {len(test_set)} examples")
print(f"Intents: {INTENTS}")

## Step 2 — Baseline: Zero-Shot Prompting with Ollama

No training — just instruct the model to classify.

In [ ]:
import ollama

CLASSIFY_PROMPT = f"""Classify the following customer message into exactly one intent category.

Categories: {', '.join(INTENTS)}

Rules:
- Output ONLY the category name, nothing else
- Use exact spelling from the list above

Message: {{message}}

Category:"""

def classify_zeroshot(message):
    prompt = CLASSIFY_PROMPT.replace("{{message}}", message)
    resp = ollama.chat(model="qwen3:4b", messages=[{"role": "user", "content": prompt}],
                       options={"temperature": 0.0, "num_predict": 20})
    raw = resp["message"]["content"].strip().lower()
    # Extract just the intent from potential verbose output
    for intent in INTENTS:
        if intent in raw:
            return intent
    return raw

# Test zero-shot on test set
print("Zero-shot classification results:\n")
zs_correct = 0
zs_start = time.time()

for text, true_label in test_set:
    pred = classify_zeroshot(text)
    match = pred == true_label
    zs_correct += int(match)
    print(f"{'✅' if match else '❌'} [{true_label:20s}] → [{pred:20s}] {text[:50]}")

zs_time = time.time() - zs_start
zs_accuracy = zs_correct / len(test_set) * 100
print(f"\nZero-shot accuracy: {zs_accuracy:.0f}% ({zs_correct}/{len(test_set)})")
print(f"Time: {zs_time:.1f}s ({zs_time/len(test_set):.2f}s per example)")

## Step 3 — Fine-Tune for Classification

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./intent_classifier_model"

SYS_MSG = f"Classify the customer message into one of: {', '.join(INTENTS)}. Output only the category name."

train_formatted = []
for text, label in train_set:
    train_formatted.append({
        "messages": [
            {"role": "system", "content": SYS_MSG},
            {"role": "user", "content": text},
            {"role": "assistant", "content": label},
        ]
    })

train_ds = Dataset.from_list(train_formatted)

trainer = SFTTrainer(
    model=MODEL_ID,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_steps=60,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_steps=5,
        max_length=256,
        bf16=True,
        logging_steps=10,
        gradient_checkpointing=True,
        report_to="none",
    ),
    train_dataset=train_ds,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

print("Training intent classifier...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("✅ Fine-tuning complete!")

## Step 4 — Evaluate Fine-Tuned Model

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

ft_model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def classify_finetuned(message):
    messages = [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": message},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=20, temperature=0.1, do_sample=True)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip().lower()
    for intent in INTENTS:
        if intent in raw:
            return intent
    return raw

print("Fine-tuned classification results:\n")
ft_correct = 0
ft_start = time.time()

for text, true_label in test_set:
    pred = classify_finetuned(text)
    match = pred == true_label
    ft_correct += int(match)
    print(f"{'✅' if match else '❌'} [{true_label:20s}] → [{pred:20s}] {text[:50]}")

ft_time = time.time() - ft_start
ft_accuracy = ft_correct / len(test_set) * 100
print(f"\nFine-tuned accuracy: {ft_accuracy:.0f}% ({ft_correct}/{len(test_set)})")
print(f"Time: {ft_time:.1f}s ({ft_time/len(test_set):.2f}s per example)")

## Step 5 — Head-to-Head Comparison

In [ ]:
print("\n" + "=" * 60)
print("PROMPTING vs FINE-TUNING COMPARISON")
print("=" * 60)
print(f"{'Metric':<25} {'Zero-Shot':>15} {'Fine-Tuned':>15}")
print("-" * 55)
print(f"{'Accuracy':<25} {zs_accuracy:>14.0f}% {ft_accuracy:>14.0f}%")
print(f"{'Total Time':<25} {zs_time:>14.1f}s {ft_time:>14.1f}s")
print(f"{'Per-example Latency':<25} {zs_time/len(test_set):>13.2f}s {ft_time/len(test_set):>13.2f}s")
print(f"{'Model Size':<25} {'~4B (Ollama)':>15} {'0.5B + LoRA':>15}")
print(f"{'Training Required':<25} {'No':>15} {'Yes (~2 min)':>15}")
print(f"{'Labeled Data Needed':<25} {'0':>15} {len(train_set):>15}")
print("=" * 60)

if ft_accuracy > zs_accuracy:
    print("\n🏆 Fine-tuning wins on accuracy!")
elif ft_accuracy == zs_accuracy:
    print("\n🤝 Tie on accuracy — prompting may be sufficient here.")
else:
    print("\n🤔 Zero-shot wins — more training data or a bigger model may help.")

## 🧠 Key Concepts Recap

| Factor | Prompting | Fine-Tuning |
|--------|-----------|-------------|
| **Setup cost** | Zero | Need labeled data + training |
| **Accuracy** | Good for clear tasks | Better for nuanced/domain tasks |
| **Latency** | Depends on model size | Can use smaller model → faster |
| **Cost at scale** | Larger model = more $ | Smaller model = cheaper |
| **Maintenance** | Update prompt anytime | Retrain when categories change |

## When to Fine-Tune vs Prompt
- **Prompt first** if you have <50 labeled examples
- **Fine-tune** if accuracy matters AND you have 200+ examples
- **Fine-tune** if you need to use a smaller/cheaper model
- **Fine-tune** if classifications are domain-specific or ambiguous